# PCA & Dimensionality Reduction: Zero to Hero

The one unsupervised method that shows up everywhere — and the one most often misread, because
its output *looks* like features and is not.

> **Prerequisites:** [`ml_foundations_zero_to_hero.ipynb`](ml_foundations_zero_to_hero.ipynb)
> for the shared workflow. [`knn_zero_to_hero.ipynb`](knn_zero_to_hero.ipynb) §1.5 is the
> problem this notebook solves — the curse of dimensionality — and
> [`svm_zero_to_hero.ipynb`](svm_zero_to_hero.ipynb) §1.5 is where the kernel trick appears,
> which §3.1 reuses.

***

## Why this one repays careful study

PCA is four lines to run and easy to get subtly wrong. The failures are quiet:

- **Its objective has two equivalent descriptions** — maximise retained variance, or minimise
  reconstruction error — and §1.2 shows they are *exactly* the same thing, not merely similar.
  Understanding that equivalence is most of understanding PCA.
- **It is unsupervised, so the biggest component need not be the useful one.** §1.8 builds a
  dataset where the top component holds **98.9% of the variance and scores 0.55** — barely
  better than a coin flip — while the 1% component classifies **perfectly**. "Keep 95% of the
  variance" would have thrown away the entire signal.
- **Components are not features.** They are directions in the original space — a *rotation*.
  §1.7 makes that concrete, and it is why "PC1 means customer value" is almost always wrong.
- **The standard leakage warning is overstated for PCA, and §3.4 measures why.** On pure noise,
  fitting PCA outside the CV loop inflates accuracy by **−0.015** — nothing. The same mistake
  with a *supervised* selector manufactures **78.5%** accuracy out of noise. Knowing which
  preprocessing steps actually leak is worth more than the blanket rule.

## Contents

| Part | What it covers |
|---|---|
| **0. Setup** | Install + imports |
| **1. Theory from zero** | The problem · **variance ⇔ reconstruction** · from scratch by eigendecomposition · **the SVD route** and why · centring and scaling · choosing k · what a component *is* · **PCA is unsupervised** |
| **2. Worked example** | Digits: compression, reconstruction, scree, and k for a downstream model |
| **3. The limits** | PCA is linear · t-SNE/UMAP are for looking only · LDA when you have labels · what actually leaks · whitening |
| **4. Tough questions** | 12 questions + 3 coding challenges |
| **5. Practice datasets** | 5 datasets with briefs |
| **6. Reading the literature** | The papers behind each section |
| **Appendix** | PCA-specific errors and a checklist |

## The one-paragraph summary

PCA finds the directions along which your data varies most, and re-expresses every point in
terms of those directions instead of the original features. Because those directions are
**orthogonal** and **ordered by variance**, you can keep the first few and discard the rest —
which is simultaneously the best possible linear compression (minimum reconstruction error) and
a way to escape the curse of dimensionality. It requires **centring**, it is meaningless without
**scaling** when your features have different units, and it is **unsupervised**: it optimises
for describing $X$, and knows nothing about $y$.

***
# Part 0 - Setup

In [ ]:
# ---------------------------------------------------------------------------
# One-time setup. Only missing packages are installed, so re-running is cheap.
# ---------------------------------------------------------------------------
import importlib.util
import subprocess
import sys

REQUIRED = [
    ("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"),
    ("scipy", "scipy"), ("sklearn", "scikit-learn"),
]
missing = [pip for mod, pip in REQUIRED if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("done")
else:
    print("all packages present")

In [ ]:
# ---------------------------------------------------------------------------
# Every import this notebook uses.
# ---------------------------------------------------------------------------
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr

from sklearn.decomposition import PCA, KernelPCA, TruncatedSVD, NMF
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE
from sklearn.datasets import (
    load_digits, load_breast_cancer, load_wine, load_iris,
    make_classification, make_circles, fetch_olivetti_faces,
)
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import (
    cross_val_score, GridSearchCV, StratifiedKFold, train_test_split,
)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["figure.dpi"] = 110

# load_digits reshapes an array in place, which NumPy 2.5 deprecates. The warning comes
# from inside scikit-learn 1.9 and there is nothing to fix on our side, so it is
# suppressed narrowly, here, once.
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=DeprecationWarning,
                            message=".*shape on a NumPy array.*")
    DIGITS = load_digits()

print("ready | numpy", np.__version__, "| pandas", pd.__version__)
print("digits loaded:", DIGITS.data.shape)

***
# Part 1 - Theory from zero

1. The problem dimensionality reduction solves
2. **Two definitions of PCA, and why they are the same**
3. The whole method from scratch
4. The SVD route, and why every library uses it
5. Centring and scaling
6. Choosing how many components
7. What a component actually *is*
8. **PCA is unsupervised** — the failure that follows

## 1.1 The problem

Three separate problems, all fixed by the same move:

- **Distances stop working.** NB-07 §1.5 measured it: as dimensions grow, the nearest and
  farthest points converge, and every distance-based method degrades.
- **Features are redundant.** Real columns are correlated — height and weight, or two adjacent
  pixels. Ten correlated columns do not carry ten columns' worth of information.
- **You cannot look at it.** Beyond three dimensions there is no plot.

The insight PCA rests on: **high-dimensional data usually does not fill its space.** It lies on
or near a much lower-dimensional surface. If the true structure is 5-dimensional, the other 59
dimensions are noise and storage cost.

In [ ]:
X_dig, y_dig = DIGITS.data, DIGITS.target
print(f"digits: {X_dig.shape[0]} images, {X_dig.shape[1]} pixels each (8x8, values 0-16)")

# Are the 64 pixel "features" independent? No - neighbouring pixels move together.
# Constant pixels have zero standard deviation, so correlation is undefined for them;
# drop those columns rather than dividing by zero.
stds = X_dig.std(axis=0)
varying = stds > 0
corr = np.corrcoef(X_dig[:, varying], rowvar=False)
m = ~np.eye(corr.shape[0], dtype=bool)
off = corr[m]
print(f"\npixels that never vary at all : {(~varying).sum()}")
print(f"pixels with std < 0.5         : {(stds < 0.5).sum()}")
print(f"\namong the {varying.sum()} varying pixels, absolute pairwise correlation:")
print(f"  mean {np.abs(off).mean():.3f}, max {np.abs(off).max():.3f}")
print(f"  pairs correlated above 0.5: {(np.abs(off) > 0.5).sum() // 2}")
print()
print("So 64 'features' is a wild overstatement of how much this data actually says. Some")
print("pixels are always zero (the border), and many others move in lockstep with their")
print("neighbours. PCA's job is to find how many directions genuinely carry information.")

## 1.2 Two definitions, one method

PCA is usually introduced one of two ways:

1. **Maximise retained variance.** Find the direction $w$ (a unit vector) along which the
   projected data $Xw$ has the largest variance. Then the next such direction orthogonal to it,
   and so on.
2. **Minimise reconstruction error.** Find the $k$-dimensional subspace that, when you project
   the data onto it and back, loses the least squared distance.

These sound like different goals. They are **the same goal**, exactly — and the reason is
Pythagoras. For centred data, each point's squared length splits cleanly into the part inside
the subspace and the part perpendicular to it:

$$ \underbrace{\lVert x \rVert^2}_{\text{total}} = \underbrace{\lVert \hat{x} \rVert^2}_{\text{kept}} + \underbrace{\lVert x - \hat{x} \rVert^2}_{\text{lost}} $$

The total is fixed by the data. So **maximising what you keep is identical to minimising what
you lose** — one subtraction apart.

In [ ]:
Xc = X_dig - X_dig.mean(axis=0)          # PCA always works on centred data (1.5)
DDOF = len(Xc) - 1
total_variance = (Xc ** 2).sum() / DDOF

print("For each k: variance kept by the top k components, and reconstruction error.\n")
print(f"{'k':>4} {'variance kept':>16} {'variance lost':>16} {'sum':>16}")
print("-" * 56)
sums = []
for k in [1, 2, 5, 10, 20, 40, 64]:
    p = PCA(n_components=k, random_state=RANDOM_STATE).fit(X_dig)
    X_recon = p.inverse_transform(p.transform(X_dig))
    kept = p.explained_variance_.sum()
    lost = ((X_dig - X_recon) ** 2).sum() / DDOF
    sums.append(kept + lost)
    print(f"{k:>4} {kept:>16.6f} {lost:>16.6f} {kept + lost:>16.6f}")

print(f"\ntotal variance in the data : {total_variance:.6f}")
print(f"largest deviation of the sum from it : {max(abs(s - total_variance) for s in sums):.2e}")
print()
print("The last column is CONSTANT, to floating-point precision, and equals the total")
print("variance of the data. Every unit of variance you keep is a unit of reconstruction")
print("error you avoid.")
print()
print("This is why you will see PCA described both ways and should not be confused: they")
print("are one objective seen from two sides. It also tells you what PCA optimises - SQUARED")
print("error - which is the same reason it is sensitive to outliers and to feature scale.")

## 1.3 The whole method from scratch

The directions that maximise projected variance are the **eigenvectors of the covariance
matrix**, and the variance along each is its **eigenvalue**.

Sketch of why: the variance of the projection $Xw$ is $w^\top \Sigma w$ where $\Sigma$ is the
covariance matrix. Maximising that subject to $\lVert w \rVert = 1$ is a Lagrange-multiplier
problem whose stationary condition is $\Sigma w = \lambda w$ — the definition of an
eigenvector. The multiplier $\lambda$ *is* the variance you get.

So the recipe is: centre → covariance → eigendecomposition → sort by eigenvalue.

In [ ]:
# PCA in five lines.
def pca_from_scratch(X, n_components=None):
    """Return (components, explained_variance, mean) via eigendecomposition."""
    mean = X.mean(axis=0)
    Xc = X - mean                                     # 1. centre
    cov = np.cov(Xc, rowvar=False)                    # 2. covariance matrix
    evals, evecs = np.linalg.eigh(cov)                # 3. eigh: symmetric -> real, ascending
    order = np.argsort(evals)[::-1]                   # 4. sort descending
    evals, evecs = evals[order], evecs[:, order]
    k = n_components or len(evals)
    return evecs.T[:k], evals[:k], mean               # 5. rows = components


comp_mine, var_mine, mean_mine = pca_from_scratch(X_dig)
sk = PCA(random_state=RANDOM_STATE).fit(X_dig)

print(f"eigenvalues vs sklearn's explained_variance_ : "
      f"max |diff| {np.abs(var_mine - sk.explained_variance_).max():.3e}")
print(f"means match                                  : "
      f"{np.allclose(mean_mine, sk.mean_)}")

# Components are only determined up to SIGN, and only where eigenvalues are distinct.
print(f"\nsmallest 5 eigenvalues: {np.round(var_mine[-5:], 10)}")
n_good = int((var_mine > 1e-6 * var_mine[0]).sum())
print(f"components with eigenvalue > 1e-6 x the largest: {n_good} of {len(var_mine)}")
print("  (digits has all-zero border pixels, so the last few eigenvalues are 0 and their")
print("   eigenvectors are an ARBITRARY basis of a degenerate subspace - not comparable.)")

mine = comp_mine[:n_good]
theirs = sk.components_[:n_good]
flip = np.sign((mine * theirs).sum(axis=1))          # align signs before comparing
print(f"\nleading {n_good} components, max |diff| after sign alignment: "
      f"{np.abs(mine * flip[:, None] - theirs).max():.3e}")
print(f"components whose sign differed from sklearn's: {int((flip < 0).sum())} of {n_good}")
print()
print("Two things worth internalising from that comparison:")
print("  1. A component and its NEGATIVE describe the same axis. Sign is arbitrary, so a")
print("     PCA plot flipping between runs or library versions means nothing.")
print("  2. Where eigenvalues are equal, individual eigenvectors are NOT determined - only")
print("     the subspace they span is. Never interpret a component whose eigenvalue is")
print("     nearly tied with its neighbour's.")

## 1.4 The SVD route

No serious implementation forms the covariance matrix. They use the **singular value
decomposition** of the centred data directly:

$$ X_c = U \Sigma V^\top $$

- The rows of $V^\top$ are the **principal components**.
- The singular values give the variances: $\lambda_i = \sigma_i^2 / (n-1)$.
- $U\Sigma$ is the transformed data — the scores.

The reason is **numerical conditioning**. Forming $X^\top X$ squares the condition number, so
you lose twice as many digits of precision as you needed to.

In [ ]:
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)

print(f"S^2/(n-1) vs explained_variance_ : max |diff| "
      f"{np.abs(S ** 2 / DDOF - sk.explained_variance_).max():.3e}")
flip2 = np.sign((Vt[:n_good] * theirs).sum(axis=1))
print(f"rows of Vt vs components_        : max |diff| "
      f"{np.abs(Vt[:n_good] * flip2[:, None] - theirs).max():.3e}")
print(f"U @ diag(S) vs transform(X)      : max |diff| "
      f"{np.abs(np.abs(U * S) - np.abs(sk.transform(X_dig))).max():.3e}")

print("\nWhy not just use the covariance matrix? Build matrices with KNOWN conditioning:\n")
rng = np.random.default_rng(0)
Q1, _ = np.linalg.qr(rng.normal(size=(400, 6)))
Q2, _ = np.linalg.qr(rng.normal(size=(6, 6)))
print(f"  {'cond(X)':>12} {'cond(X.T @ X)':>16} {'ratio':>12}")
print("  " + "-" * 42)
for spread in [1e3, 1e6, 1e8]:
    M = Q1 @ np.diag(np.logspace(0, -np.log10(spread), 6)) @ Q2.T
    cX, cG = np.linalg.cond(M), np.linalg.cond(M.T @ M)
    print(f"  {cX:>12.2e} {cG:>16.2e} {cG / cX:>12.2e}")

print()
print("cond(X.T @ X) = cond(X)^2, exactly. float64 carries about 16 digits, so a matrix")
print("with condition number 1e8 - not unusual - leaves the covariance matrix at 1e16 and")
print("effectively no precision at all.")
print()
print("SVD never forms X.T @ X, so it keeps the smaller number. That is the whole reason")
print("sklearn's PCA is implemented on top of it, and the reason to reach for")
print("`TruncatedSVD` on sparse data (it skips centring, which would destroy sparsity).")

## 1.5 Centring and scaling

**Centring is mandatory**, and sklearn does it for you. If you write the SVD yourself and skip
it, the first component points at *the mean* rather than at the direction of greatest spread.

**Scaling is a modelling decision**, and sklearn does *not* do it for you. PCA maximises
variance, and variance has units. A feature measured in grams has a thousand times the variance
of the same feature in kilograms — and PCA will hand it the first component for that reason
alone.

In [ ]:
rng = np.random.default_rng(0)
# an elongated cloud, far from the origin
cloud = rng.normal(size=(300, 2)) @ np.array([[3.0, 0.0], [0.0, 0.4]]) + np.array([50.0, 50.0])

p_ok = PCA(n_components=2).fit(cloud)                       # sklearn centres
_, _, Vt_bad = np.linalg.svd(cloud, full_matrices=False)    # forgot to centre
mean_dir = cloud.mean(axis=0) / np.linalg.norm(cloud.mean(axis=0))

print("A. CENTRING\n")
print(f"  data mean                    : {cloud.mean(axis=0).round(2)}")
print(f"  PC1, correctly centred       : {p_ok.components_[0].round(4)}")
print(f"  PC1 from UNCENTRED svd       : {Vt_bad[0].round(4)}")
print(f"  unit vector toward the mean  : {mean_dir.round(4)}")
ang = np.degrees(np.arccos(np.clip(abs(np.dot(Vt_bad[0], mean_dir)), 0, 1)))
print(f"\n  angle between uncentred PC1 and the mean direction: {ang:.2f} degrees")
print("  Without centring, the 'principal component' is just the direction of the mean.")
print("  sklearn protects you here; hand-rolled SVD does not.")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, (title, origin, vecs) in zip(axes, [
        ("centred (correct)", cloud.mean(axis=0), p_ok.components_),
        ("uncentred (wrong)", np.zeros(2), Vt_bad)]):
    ax.scatter(cloud[:, 0], cloud[:, 1], s=8, alpha=0.4)
    for v, scale in zip(vecs, [12, 4]):
        ax.arrow(origin[0], origin[1], v[0] * scale, v[1] * scale,
                 head_width=1.0, color="crimson", lw=1.5)
    ax.scatter(*origin, color="black", zorder=5, s=25)
    ax.set_title(title, fontsize=10)
    ax.set_aspect("equal")
fig.tight_layout(); plt.show()

In [ ]:
print("B. SCALING\n")
wine = load_wine()
Xw, yw = wine.data, wine.target

p_raw = PCA(n_components=2).fit(Xw)
p_sc = PCA(n_components=2).fit(StandardScaler().fit_transform(Xw))

print(f"  PC1 explained variance ratio, RAW    : {p_raw.explained_variance_ratio_[0]:.4f}")
print(f"  PC1 explained variance ratio, SCALED : {p_sc.explained_variance_ratio_[0]:.4f}")

load_raw = np.abs(p_raw.components_[0])
top_raw = int(load_raw.argmax())
print(f"\n  RAW PC1 is essentially one feature: '{wine.feature_names[top_raw]}' holds "
      f"{load_raw[top_raw] / load_raw.sum():.1%} of the loading")
load_sc = np.abs(p_sc.components_[0])
top_sc = int(load_sc.argmax())
print(f"  SCALED PC1 top feature: '{wine.feature_names[top_sc]}' with "
      f"{load_sc[top_sc] / load_sc.sum():.1%} - spread across many features")

variances = Xw.var(axis=0)
print(f"\n  raw feature variances span {variances.min():.4f} to {variances.max():,.0f} "
      f"({variances.max() / variances.min():,.0f}x)")

skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
print(f"\n  {'pipeline':<46} {'CV accuracy':>12}")
print("  " + "-" * 60)
for label, model in [
    ("PCA(2) -> logistic regression  (no scaler)",
     make_pipeline(PCA(n_components=2, random_state=RANDOM_STATE),
                   LogisticRegression(max_iter=3000))),
    ("scale -> PCA(2) -> logistic regression",
     make_pipeline(StandardScaler(), PCA(n_components=2, random_state=RANDOM_STATE),
                   LogisticRegression(max_iter=3000))),
]:
    print(f"  {label:<46} {cross_val_score(model, Xw, yw, cv=skf).mean():>12.4f}")

print()
print("Unscaled, PC1 reports 99.8% of the variance and is a single chemical measurement")
print("wearing a component's clothes. The downstream model loses about a quarter of its")
print("accuracy.")
print()
print("When to scale: whenever features have different UNITS. When not to: when they are")
print("already commensurable - pixels on one 0-16 scale, or one gene-expression assay -")
print("where scaling would amplify the quietest, noisiest channels.")

## 1.6 Choosing how many components

Three approaches, in increasing order of how much you should trust them:

- **The scree plot** — plot eigenvalues and look for the "elbow" where they flatten. Subjective.
- **A cumulative-variance threshold** — keep enough components for 95% of the variance.
  `PCA(n_components=0.95)` does this for you.
- **Cross-validate it.** If you have labels and a downstream model, $k$ is a hyperparameter
  like any other, and the only criterion that matters is the model's score.

In [ ]:
full = PCA(random_state=RANDOM_STATE).fit(X_dig)
evr = full.explained_variance_ratio_
cum = np.cumsum(evr)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(range(1, 65), evr, marker="o", markersize=3)
axes[0].set(xlabel="component", ylabel="variance explained", title="Scree plot")
axes[0].grid(alpha=0.3)
axes[1].plot(range(1, 65), cum, marker="o", markersize=3)
for t, c in [(0.8, "gray"), (0.9, "orange"), (0.95, "crimson")]:
    axes[1].axhline(t, color=c, ls=":", lw=1)
    axes[1].text(64, t, f" {t:.0%}", va="center", fontsize=7, color=c)
axes[1].set(xlabel="components kept", ylabel="cumulative variance",
            title="Cumulative explained variance")
axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

for t in [0.80, 0.90, 0.95, 0.99]:
    print(f"  {t:.0%} of the variance needs {int(np.searchsorted(cum, t) + 1):>2} of 64 components")
auto = PCA(n_components=0.95, random_state=RANDOM_STATE).fit(X_dig)
print(f"\n  PCA(n_components=0.95) selects {auto.n_components_} components - the same answer, "
      f"in one argument")
print()
print("The scree plot's 'elbow' here is not sharp. That is normal, and it is why the elbow")
print("method is a starting point rather than an answer.")

In [ ]:
# Does the variance rule pick the k your MODEL wants?
print(f"{'k':>4} {'cumulative variance':>21} {'KNN CV':>9} {'logreg CV':>11}")
print("-" * 48)
scores = {}
for k in [2, 5, 10, 20, 29, 40, 64]:
    knn = cross_val_score(make_pipeline(PCA(n_components=k, random_state=RANDOM_STATE),
                                        KNeighborsClassifier()), X_dig, y_dig, cv=skf).mean()
    lr = cross_val_score(make_pipeline(StandardScaler(),
                                       PCA(n_components=k, random_state=RANDOM_STATE),
                                       LogisticRegression(max_iter=5000)),
                         X_dig, y_dig, cv=skf).mean()
    scores[k] = (knn, lr)
    print(f"{k:>4} {cum[k-1]:>20.1%} {knn:>9.4f} {lr:>11.4f}")

best_knn = max(scores, key=lambda k: scores[k][0])
print(f"\nbest k for KNN by CV : {best_knn} (accuracy {scores[best_knn][0]:.4f})")
print(f"the 95% variance rule said : {auto.n_components_} "
      f"(accuracy {scores[29][0]:.4f})")
print(f"difference : {scores[best_knn][0] - scores[29][0]:+.4f}")
print()
print("On this dataset the 95% rule lands within a rounding error of the cross-validated")
print("optimum, so it was a perfectly good shortcut. Do not over-generalise that - it is a")
print("heuristic about the DATA's variance, and it has no way to know what your model needs.")
print()
print("Note the shape of the KNN column: it climbs steeply to about k=20 and is then FLAT.")
print("Dropping the last 24 components costs nothing measurable - the peak at k=40 beats")
print("k=64 by 0.0011, which is a fifth of one test fold's worth of a single sample and")
print("should not be called an improvement.")
print()
print("The honest reading is 'those components carried no usable signal', not 'PCA denoised")
print("the data'. That is still a useful result: it says you can halve the dimensionality")
print("for free. Just do not inflate 'free' into 'better' - 2.4 tests that properly.")

## 1.7 What a component actually is

The single most common misreading: treating components as new *features* with meanings.

A component is a **direction in the original feature space** — a unit vector with one
coefficient (a **loading**) per original feature. The transformed data is the original data
expressed in a rotated coordinate system. PCA does not create anything; it re-describes.

Three consequences:

- **Components are combinations of every feature**, so "PC1 = customer value" is at best a
  summary of a weighted sum of all your columns.
- **PCA does not select features.** All 64 pixels are still needed at prediction time to
  compute PC1. If your goal is to *stop collecting* features, PCA is the wrong tool — you want
  feature selection.
- **Interpretation requires reading the loadings**, and is only safe when eigenvalues are
  well separated (§1.3).

In [ ]:
p2 = PCA(n_components=2, random_state=RANDOM_STATE).fit(X_dig)
print(f"components_ shape : {p2.components_.shape}   (2 components x 64 original pixels)")
print(f"each component is a unit vector: norms = {np.linalg.norm(p2.components_, axis=1).round(6)}")
print(f"and they are orthogonal: PC1 . PC2 = {np.dot(*p2.components_):.2e}")

# A component over pixels is itself an image - the clearest possible view of "direction".
fig, axes = plt.subplots(1, 6, figsize=(12, 2.3))
axes[0].imshow(X_dig.mean(axis=0).reshape(8, 8), cmap="gray")
axes[0].set_title("mean image", fontsize=8)
p6 = PCA(n_components=5, random_state=RANDOM_STATE).fit(X_dig)
for i, ax in enumerate(axes[1:]):
    ax.imshow(p6.components_[i].reshape(8, 8), cmap="RdBu")
    ax.set_title(f"PC{i+1}\n{p6.explained_variance_ratio_[i]:.1%}", fontsize=8)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); plt.show()

print("Red and blue are opposite signs. Each component is a PATTERN of pixel intensities:")
print("'more of this region, less of that one'. An image is rebuilt as the mean image plus")
print("a weighted sum of these patterns.")
print()

# the reconstruction identity, checked
img = X_dig[0]
z = p6.transform(img.reshape(1, -1))[0]
manual = p6.mean_ + z @ p6.components_
print(f"mean + sum(score_i * PC_i) reproduces inverse_transform: "
      f"{np.allclose(manual, p6.inverse_transform(z.reshape(1, -1))[0])}")
print()
print("PCA needs ALL 64 pixels to compute those 5 scores. It has reduced the DIMENSIONALITY")
print("of the representation, not the number of measurements you must collect.")

## 1.8 PCA is unsupervised

PCA never sees $y$. It finds the directions that best describe $X$ — and there is no reason in
principle for the direction of greatest variance to be the direction that separates your
classes.

Usually they are related: whatever drives variation often drives the label too. But "usually"
is not "always", and when it fails, it fails silently and completely.

In [ ]:
# Build the failure deliberately: one huge-variance direction with no signal, and one
# small-variance direction that separates the classes perfectly.
rng = np.random.default_rng(0)
n = 600
y_syn = rng.integers(0, 2, n)
noise_axis = rng.normal(0, 10.0, n)                                  # big variance, no signal
signal_axis = np.where(y_syn == 1, 1.0, -1.0) + rng.normal(0, 0.25, n)   # small, all signal
X_syn = np.column_stack([noise_axis, signal_axis])

p = PCA(n_components=2, random_state=RANDOM_STATE).fit(X_syn)
Z = p.transform(X_syn)
print(f"explained variance ratio : {p.explained_variance_ratio_.round(4)}")
print()
print(f"{'kept':<34} {'CV accuracy':>12}")
print("-" * 48)
for label, Zi in [("PC1 only  (99% of the variance)", Z[:, [0]]),
                  ("PC2 only  (1% of the variance)", Z[:, [1]]),
                  ("both original features", X_syn)]:
    print(f"{label:<34} {cross_val_score(LogisticRegression(), Zi, y_syn, cv=skf).mean():>12.4f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].scatter(X_syn[:, 0], X_syn[:, 1], c=y_syn, cmap="coolwarm", s=8, alpha=0.6)
axes[0].set(title="the data (note the axis scales)", xlabel="noise feature",
            ylabel="signal feature")
axes[1].scatter(Z[:, 0], Z[:, 1], c=y_syn, cmap="coolwarm", s=8, alpha=0.6)
axes[1].set(title="after PCA", xlabel="PC1 (99% of variance)", ylabel="PC2 (1%)")
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

print("PCA ranked the useless direction FIRST, because it has more variance. Keeping '95% of")
print("the variance' would have kept only PC1 and thrown the entire signal away.")
print()
print("Nothing warns you. The explained-variance report looks excellent - 99% in one")
print("component is exactly what a textbook success looks like.")

In [ ]:
# Does this happen on real data, or only in constructed examples? Test it honestly.
Xb, yb = load_breast_cancer(return_X_y=True)
Xb_s = StandardScaler().fit_transform(Xb)
pb = PCA(n_components=10, random_state=RANDOM_STATE).fit(Xb_s)
Zb = pb.transform(Xb_s)

print("breast cancer: each component's variance share vs its accuracy used ALONE\n")
print(f"{'PC':>4} {'variance share':>16} {'accuracy alone':>17}")
print("-" * 40)
solo = []
for i in range(10):
    s = cross_val_score(LogisticRegression(max_iter=3000), Zb[:, [i]], yb, cv=skf).mean()
    solo.append(s)
    print(f"{i+1:>4} {pb.explained_variance_ratio_[i]:>15.1%} {s:>17.4f}")

rho = spearmanr(pb.explained_variance_ratio_, solo).statistic
print(f"\nSpearman correlation between variance share and predictiveness: {rho:.4f}")
print()
print("Here PC1 IS the most predictive component, and the correlation is positive - so the")
print("common case really is that variance and usefulness line up. Note though that PC2")
print(f"holds {pb.explained_variance_ratio_[1]:.1%} of the variance and is beaten by PC3, which holds "
      f"{pb.explained_variance_ratio_[2]:.1%}.")
print()
print("The honest summary: PCA is usually a reasonable default, and occasionally it")
print("silently deletes your signal. The cheap insurance is to CHECK - cross-validate k")
print("against your actual model rather than trusting a variance threshold, and if you have")
print("labels and want a supervised reduction, use LDA (3.3).")

***
# Part 2 - Worked example: compressing handwritten digits

**The task.** 1,797 handwritten digits as 8×8 grayscale images — 64 pixel features, values 0
to 16. How few numbers can represent one, and what does the reconstruction look like?

**Why this dataset.** Because you can *see* the answer. Every intermediate step is an image,
so a claim like "10 components keep 74% of the variance" stops being abstract.

## 2.1 Look at the data

In [ ]:
print(f"{X_dig.shape[0]} images, {X_dig.shape[1]} pixels, {len(np.unique(y_dig))} classes")
print(f"pixel value range: {X_dig.min():.0f} to {X_dig.max():.0f}")
print(f"class balance: {np.bincount(y_dig)}")

fig, axes = plt.subplots(2, 10, figsize=(12, 2.6))
for digit in range(10):
    members = np.where(y_dig == digit)[0][:2]
    for row, idx in enumerate(members):
        axes[row, digit].imshow(X_dig[idx].reshape(8, 8), cmap="gray")
        axes[row, digit].set_xticks([]); axes[row, digit].set_yticks([])
    axes[0, digit].set_title(str(digit), fontsize=9)
fig.suptitle("two examples of each digit", fontsize=9)
fig.tight_layout(); plt.show()

print("All features share one 0-16 scale, so this is the case where scaling PCA is a real")
print("decision rather than an obvious yes (1.5). We test both in 2.4.")

## 2.2 Compression and reconstruction

The whole point, made visible: project down to $k$ dimensions, project back, and look.

In [ ]:
print(f"{'k':>4} {'variance kept':>15} {'recon RMSE':>12} {'numbers per image':>19} "
      f"{'storage vs raw':>16}")
print("-" * 72)
for k in [1, 2, 5, 10, 20, 29, 40, 64]:
    p = PCA(n_components=k, random_state=RANDOM_STATE).fit(X_dig)
    X_rec = p.inverse_transform(p.transform(X_dig))
    rmse = np.sqrt(((X_dig - X_rec) ** 2).mean())
    stored = len(X_dig) * k + k * 64          # the codes AND the basis you must ship
    print(f"{k:>4} {p.explained_variance_ratio_.sum():>14.1%} {rmse:>12.4f} {k:>19} "
          f"{len(X_dig) * 64 / stored:>15.2f}x")

print("\nPixel values run 0-16, so an RMSE around 2 is a visibly imperfect but readable digit.")
print("Note the last row: at k=64 you keep everything and storage is 0.97x - WORSE than raw,")
print("because you also have to store the 64x64 basis. Compression is only a saving when k")
print("is genuinely small relative to the number of samples.")

In [ ]:
# The same thing, as images.
sample_idx = [np.where(y_dig == d)[0][0] for d in [0, 3, 5, 8, 9]]
ks = [1, 2, 5, 10, 20, 40, 64]

fig, axes = plt.subplots(len(sample_idx), len(ks) + 1, figsize=(11, 7))
for row, idx in enumerate(sample_idx):
    axes[row, 0].imshow(X_dig[idx].reshape(8, 8), cmap="gray", vmin=0, vmax=16)
    axes[row, 0].set_ylabel(f"digit {y_dig[idx]}", fontsize=8)
    if row == 0:
        axes[row, 0].set_title("original", fontsize=8)
    for col, k in enumerate(ks, start=1):
        p = PCA(n_components=k, random_state=RANDOM_STATE).fit(X_dig)
        rec = p.inverse_transform(p.transform(X_dig[idx].reshape(1, -1)))[0]
        axes[row, col].imshow(rec.reshape(8, 8), cmap="gray", vmin=0, vmax=16)
        if row == 0:
            axes[row, col].set_title(f"k={k}", fontsize=8)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("PCA reconstruction as components are added", fontsize=10)
fig.tight_layout(); plt.show()

print("k=1 gives a blurred average - every digit looks like the same smudge, because one")
print("number cannot distinguish ten classes.")
print("By k=10 the digits are readable. By k=20 they are nearly indistinguishable from the")
print("originals, on a third of the storage.")
print()
print("This is the intuition to keep: the 64 pixels were never 64 independent quantities.")
print("The images live near a ~20-dimensional surface inside a 64-dimensional space.")

## 2.3 The 2-D picture

Two components is almost always too few to model with, but it is the only way to *see* the
whole dataset at once.

In [ ]:
p2 = PCA(n_components=2, random_state=RANDOM_STATE)
Z2 = p2.fit_transform(X_dig)

fig, ax = plt.subplots(figsize=(6.5, 5))
sc = ax.scatter(Z2[:, 0], Z2[:, 1], c=y_dig, cmap="tab10", s=8, alpha=0.7)
ax.set(xlabel=f"PC1 ({p2.explained_variance_ratio_[0]:.1%} of variance)",
       ylabel=f"PC2 ({p2.explained_variance_ratio_[1]:.1%})",
       title="Digits in the first two principal components")
fig.colorbar(sc, ax=ax, ticks=range(10), label="digit")
ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

print(f"These two components hold {p2.explained_variance_ratio_.sum():.1%} of the variance.")
print()
print("Some digits separate cleanly (0 and 4 sit at opposite ends); others overlap heavily.")
print("Do NOT conclude that the overlapping ones are inseparable - you are looking at 28% of")
print("the information. 2.4 shows a model using more components doing far better.")
print()
print("This is the most common misuse of a PCA scatter plot: reading 'the classes overlap'")
print("off a 2-D projection and concluding the problem is hard.")

## 2.4 PCA in a supervised pipeline

The practical question: does putting PCA in front of a classifier help, and at what $k$?

Everything goes inside a `Pipeline` so the rotation is fitted on training folds only. §3.4
measures how much that actually matters for PCA specifically — the answer is more interesting
than the usual warning.

In [ ]:
print(f"{'pipeline':<48} {'CV accuracy':>12} {'fit (s)':>9}")
print("-" * 72)
for label, model in [
    ("KNN on all 64 pixels", KNeighborsClassifier()),
    ("PCA(10) -> KNN", make_pipeline(PCA(n_components=10, random_state=RANDOM_STATE),
                                     KNeighborsClassifier())),
    ("PCA(29) -> KNN   [the 95% rule]",
     make_pipeline(PCA(n_components=29, random_state=RANDOM_STATE), KNeighborsClassifier())),
    ("PCA(40) -> KNN", make_pipeline(PCA(n_components=40, random_state=RANDOM_STATE),
                                     KNeighborsClassifier())),
    ("logistic regression on all 64", make_pipeline(StandardScaler(),
                                                    LogisticRegression(max_iter=5000))),
    ("scale -> PCA(29) -> logistic regression",
     make_pipeline(StandardScaler(), PCA(n_components=29, random_state=RANDOM_STATE),
                   LogisticRegression(max_iter=5000))),
    ("random forest on all 64",
     RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
]:
    t0 = time.time()
    s = cross_val_score(model, X_dig, y_dig, cv=skf).mean()
    print(f"{label:<48} {s:>12.4f} {time.time() - t0:>9.2f}")

print()
print("Read this table sceptically - it does not say what a PCA tutorial usually says.")
print()
print("PCA(29) TIES with using all 64 pixels, and PCA(10) is clearly worse. Only at k=40 is")
print("there a gain, and it is about +0.001 - well inside noise. So on digits, PCA before")
print("KNN neither helps nor hurts accuracy.")
print()
print("Nor does it save time HERE: fitting PCA costs more than it saves on 1,797 rows of 64")
print("features. The speed argument is real but needs scale - thousands of features, or a")
print("model whose cost grows faster than linearly in dimension.")
print()
print("The honest case for PCA in a supervised pipeline is therefore narrower than it is")
print("usually sold: it buys denoising when the trailing components really are noise, and")
print("speed when the dimensionality is genuinely large. On a small, clean, already-compact")
print("dataset it mostly buys nothing - and that is a perfectly good result to report.")

In [ ]:
# Tune k properly, as a hyperparameter.
pipe = Pipeline([("pca", PCA(random_state=RANDOM_STATE)),
                 ("knn", KNeighborsClassifier())])
grid = {"pca__n_components": [5, 10, 15, 20, 30, 40, 50],
        "pca__whiten": [False, True],
        "knn__n_neighbors": [1, 3, 5, 9]}
search = GridSearchCV(pipe, grid, cv=skf, n_jobs=-1)
search.fit(X_dig, y_dig)

print(f"combinations tried : {len(search.cv_results_['params'])}")
print(f"best CV accuracy   : {search.best_score_:.4f}")
print(f"best parameters    : {search.best_params_}")

res = pd.DataFrame(search.cv_results_)
cols = ["param_pca__n_components", "param_pca__whiten", "param_knn__n_neighbors",
        "mean_test_score", "std_test_score"]
top = res.nlargest(5, "mean_test_score")[cols]
top.columns = ["k", "whiten", "n_neighbors", "mean CV", "std CV"]
print(f"\ntop 5 of {len(res)}:")
print(top.to_string(index=False))

print("\nbest score at each k (over whitening and n_neighbors):")
for k in sorted(res["param_pca__n_components"].unique()):
    sub = res[res["param_pca__n_components"] == k]
    print(f"  k={k:<3} {sub['mean_test_score'].max():.4f}")
print()
print("The curve is flat from about k=30 onward - the same pattern NB-07 found for its own")
print("k, and the same lesson: the search has established a PLATEAU, not a winner.")
print()
print("Be careful comparing this best score against the 'KNN on all 64 pixels' row above:")
print("this search also tuned n_neighbors, so it is not a like-for-like PCA-vs-no-PCA")
print("comparison. Tuning two things and crediting the win to one of them is an easy way")
print("to talk yourself into a preprocessing step you do not need.")

## 2.5 The test set, once

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X_dig, y_dig, test_size=0.25,
                                      stratify=y_dig, random_state=RANDOM_STATE)
best = search.best_estimator_.fit(Xtr, ytr)
pred = best.predict(Xte)

print(f"test accuracy : {accuracy_score(yte, pred):.4f}")
print(f"CV accuracy   : {search.best_score_:.4f}")

cm = confusion_matrix(yte, pred)
print("\nconfusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(cm, index=[f"{d}" for d in range(10)],
                   columns=[f"{d}" for d in range(10)]).to_string())

errors = np.where(pred != yte)[0]
print(f"\n{len(errors)} misclassified of {len(yte)}")
if len(errors):
    fig, axes = plt.subplots(1, min(8, len(errors)), figsize=(10, 1.8))
    axes = np.atleast_1d(axes)
    for ax, e in zip(axes, errors[:8]):
        ax.imshow(Xte[e].reshape(8, 8), cmap="gray")
        ax.set_title(f"{yte[e]}->{pred[e]}", fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle("mistakes (true -> predicted)", fontsize=9)
    fig.tight_layout(); plt.show()
    print("Look at them: most are genuinely ambiguous handwriting. At this accuracy the")
    print("remaining errors are largely label noise rather than model failure.")

***
# Part 3 - The limits

Four things PCA is not, and one thing everyone warns about that turns out to be smaller than
advertised.

## 3.1 PCA is linear — it cannot unfold curved structure

PCA rotates the axes. A rotation cannot make a non-linearly-separable problem separable,
however many components you keep, because rotation preserves every distance and angle.

In [ ]:
X_c, y_c = make_circles(n_samples=600, factor=0.35, noise=0.06, random_state=RANDOM_STATE)

print(f"{'representation':<42} {'linear model CV':>16} {'variance kept':>15}")
print("-" * 76)
p_full = PCA(n_components=2, random_state=RANDOM_STATE).fit(X_c)
for label, Z, vk in [
    ("raw 2-D", X_c, 1.0),
    ("PCA(2) - a pure rotation", p_full.transform(X_c), p_full.explained_variance_ratio_.sum()),
    ("PCA(1)", PCA(n_components=1, random_state=RANDOM_STATE).fit_transform(X_c),
     PCA(n_components=1, random_state=RANDOM_STATE).fit(X_c).explained_variance_ratio_.sum()),
    ("KernelPCA(RBF, 2)", KernelPCA(n_components=2, kernel="rbf", gamma=2.0,
                                    random_state=RANDOM_STATE).fit_transform(X_c), np.nan),
]:
    s = cross_val_score(LogisticRegression(), Z, y_c, cv=skf).mean()
    vk_s = f"{vk:>14.1%}" if np.isfinite(vk) else f"{'n/a':>15}"
    print(f"{label:<42} {s:>16.4f} {vk_s}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, (title, Z) in zip(axes, [
        ("original", X_c),
        ("PCA(2): rotated, still nested", p_full.transform(X_c)),
        ("KernelPCA(RBF): now separable",
         KernelPCA(n_components=2, kernel="rbf", gamma=2.0,
                   random_state=RANDOM_STATE).fit_transform(X_c))]):
    ax.scatter(Z[:, 0], Z[:, 1], c=y_c, cmap="coolwarm", s=8, alpha=0.7)
    ax.set_title(title, fontsize=9); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

print("PCA(2) keeps 100% of the variance and changes the linear model's score by nothing at")
print("all. It could not have done otherwise: it rotated the picture.")
print()
print("KernelPCA applies the kernel trick from NB-06 section 1.5 - PCA in an implicit")
print("high-dimensional space - and the circles become linearly separable. The cost is the")
print("same as any kernel method: an n x n matrix, so it does not scale, and gamma is now a")
print("hyperparameter you must tune.")

## 3.2 t-SNE and UMAP are for looking, not for modelling

They are **manifold learning** methods that optimise a very different objective: keep points
that were close together close, and let everything else go. That makes beautiful plots and
terrible pipeline components.

In [ ]:
rng = np.random.default_rng(0)
idx = rng.choice(len(X_dig), 600, replace=False)
Xs = X_dig[idx]
ys = y_dig[idx]

Z_pca = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xs)
tsne = TSNE(n_components=2, random_state=RANDOM_STATE, init="pca", perplexity=30)
Z_tsne = tsne.fit_transform(Xs)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, Z) in zip(axes, [("PCA(2)", Z_pca), ("t-SNE(2)", Z_tsne)]):
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=ys, cmap="tab10", s=10, alpha=0.8)
    ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(sc, ax=axes, ticks=range(10), label="digit", shrink=0.8)
plt.show()

def neighbourhood_kept(X_hi, X_lo, k=10):
    """Fraction of each point's k true nearest neighbours that survive the embedding."""
    hi = NearestNeighbors(n_neighbors=k + 1).fit(X_hi).kneighbors(return_distance=False)
    lo = NearestNeighbors(n_neighbors=k + 1).fit(X_lo).kneighbors(return_distance=False)
    return np.mean([len(set(a) & set(b)) / k for a, b in zip(hi, lo)])

d_hi = pdist(Xs)
print(f"{'embedding':<12} {'LOCAL: 10-NN kept':>20} {'GLOBAL: distance Spearman':>27}")
print("-" * 62)
for label, Z in [("PCA(2)", Z_pca), ("t-SNE(2)", Z_tsne)]:
    print(f"{label:<12} {neighbourhood_kept(Xs, Z):>19.1%} "
          f"{spearmanr(pdist(Z), d_hi).statistic:>27.4f}")

print()
print("That is the trade, measured. t-SNE preserves roughly three times as many true near")
print("neighbours as PCA - its clusters are real. But it is WORSE at global distances, which")
print("is why you must not read anything into the gaps between clusters or their sizes.")

In [ ]:
print("Three more reasons t-SNE cannot be a preprocessing step:\n")

print(f"1. There is no transform() for new data: hasattr(tsne, 'transform') = "
      f"{hasattr(tsne, 'transform')}")
print("   You cannot project a new point into an existing embedding, so there is no")
print("   train/serve boundary at all. This alone settles it.\n")

print("2. The result depends on perplexity, which is not a nuisance parameter:")
for perp in [5, 30, 50]:
    Zp = TSNE(n_components=2, random_state=RANDOM_STATE, init="pca",
              perplexity=perp).fit_transform(Xs)
    print(f"     perplexity={perp:<3} 10-NN kept {neighbourhood_kept(Xs, Zp):>6.1%}   "
          f"global Spearman {spearmanr(pdist(Zp), d_hi).statistic:>7.4f}")

print("\n3. It is slow, and it scales badly:")
for m in [300, 600, 1200]:
    t0 = time.time()
    TSNE(n_components=2, random_state=RANDOM_STATE, init="pca",
         perplexity=30).fit_transform(X_dig[:m])
    t_ts = time.time() - t0
    t0 = time.time()
    PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_dig[:m])
    t_p = time.time() - t0
    print(f"     n={m:<5} t-SNE {t_ts:>6.2f}s   PCA {t_p:>6.4f}s   "
          f"({t_ts / max(t_p, 1e-9):>6.0f}x)")

print()
print("Use t-SNE (or UMAP, which is faster and does offer a transform) to LOOK at data -")
print("to find clusters worth investigating, or to spot a labelling error. Then model on")
print("PCA components or the raw features.")
print()
print("And when you read a t-SNE plot in a paper: the clusters mean something, the distances")
print("between them do not, and the shapes are an artifact of the optimisation.")

## 3.3 If you have labels, PCA is the wrong tool

**Linear Discriminant Analysis** solves the supervised version of the same problem: find the
directions that best *separate the classes*, rather than those that best describe the data.

In [ ]:
Xb, yb = load_breast_cancer(return_X_y=True)

print(f"{'reduction to 1 dimension':<44} {'CV accuracy':>12}")
print("-" * 58)
for label, model in [
    ("PCA(1) -> logistic regression   [unsupervised]",
     make_pipeline(StandardScaler(), PCA(n_components=1, random_state=RANDOM_STATE),
                   LogisticRegression(max_iter=3000))),
    ("LDA(1) -> logistic regression   [supervised]",
     make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(n_components=1),
                   LogisticRegression(max_iter=3000))),
]:
    print(f"{label:<44} {cross_val_score(model, Xb, yb, cv=skf).mean():>12.4f}")

print(f"\nLDA is limited to at most (n_classes - 1) components = "
      f"{len(np.unique(yb)) - 1} here.")
print("PCA has no such limit, which is why it is still the right choice when you have many")
print("classes, no labels at all, or you want more components than LDA can give you.")
print()
print("The rule: reducing dimensions FOR A SUPERVISED TASK and you have the labels? Try LDA")
print("first. Exploring, compressing, denoising, or preparing input for an unsupervised")
print("method? PCA.")

## 3.4 What actually leaks

Every tutorial says "put PCA inside the Pipeline or you leak". Putting it inside the pipeline
is correct — but the *reason* usually given is wrong, and the difference matters when you are
deciding what to check in someone else's code.

The clean test is data with **no signal at all**: labels independent of features. Honest
cross-validation must return chance. Anything above chance is exactly the size of the leak.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n_s, p_s, k_s = 200, 2000, 20
X_noise = rng.normal(size=(n_s, p_s))
y_noise = rng.integers(0, 2, n_s)          # independent of X: the truth is 0.50

print(f"pure noise: {n_s} samples x {p_s} features, y independent of X")
print(f"honest accuracy must be ~0.50\n")
print(f"{'transform':<44} {'fitted on ALL':>14} {'in Pipeline':>13} {'inflation':>11}")
print("-" * 84)

Z = PCA(n_components=k_s, random_state=RANDOM_STATE).fit_transform(
    StandardScaler().fit_transform(X_noise))
leak_pca = cross_val_score(LogisticRegression(max_iter=3000), Z, y_noise, cv=skf).mean()
ok_pca = cross_val_score(make_pipeline(StandardScaler(),
                                       PCA(n_components=k_s, random_state=RANDOM_STATE),
                                       LogisticRegression(max_iter=3000)),
                         X_noise, y_noise, cv=skf).mean()
print(f"{'PCA  [unsupervised]':<44} {leak_pca:>14.4f} {ok_pca:>13.4f} "
      f"{leak_pca - ok_pca:>+11.4f}")

Zs = SelectKBest(f_classif, k=k_s).fit_transform(X_noise, y_noise)
leak_sel = cross_val_score(LogisticRegression(max_iter=3000), Zs, y_noise, cv=skf).mean()
ok_sel = cross_val_score(make_pipeline(SelectKBest(f_classif, k=k_s),
                                       LogisticRegression(max_iter=3000)),
                         X_noise, y_noise, cv=skf).mean()
print(f"{'SelectKBest(f_classif)  [SUPERVISED]':<44} {leak_sel:>14.4f} {ok_sel:>13.4f} "
      f"{leak_sel - ok_sel:>+11.4f}")

Zl = LinearDiscriminantAnalysis(n_components=1).fit_transform(X_noise, y_noise)
leak_lda = cross_val_score(LogisticRegression(max_iter=3000), Zl, y_noise, cv=skf).mean()
ok_lda = cross_val_score(make_pipeline(LinearDiscriminantAnalysis(n_components=1),
                                       LogisticRegression(max_iter=3000)),
                         X_noise, y_noise, cv=skf).mean()
print(f"{'LDA  [SUPERVISED]':<44} {leak_lda:>14.4f} {ok_lda:>13.4f} "
      f"{leak_lda - ok_lda:>+11.4f}")

print()
print("Read that carefully, because it corrects a very widely repeated claim.")
print()
print("PCA fitted on all the data inflates the score by essentially NOTHING. It cannot")
print("smuggle label information across the split, because it never sees the labels. The")
print("only thing it borrows is the test set's FEATURE distribution.")
print()
print("The supervised transforms fabricate accuracy out of pure noise - SelectKBest turns a")
print("coin flip into a confident-looking classifier. THAT is what preprocessing leakage")
print("looks like when it is real.")

In [ ]:
# So should PCA go in the Pipeline? Yes - for reasons that are not about inflated CV.
print("Does the small PCA difference at least have a consistent sign? Vary the sample size:\n")
print(f"  {'n':>6} {'PCA fitted on all':>19} {'PCA in Pipeline':>17} {'difference':>12}")
print("  " + "-" * 58)
for n_r in [50, 100, 300, 1000]:
    Xr, yr = make_classification(n_samples=n_r, n_features=200, n_informative=10,
                                 n_redundant=20, random_state=RANDOM_STATE)
    Zr = PCA(n_components=10, random_state=RANDOM_STATE).fit_transform(
        StandardScaler().fit_transform(Xr))
    lk = cross_val_score(LogisticRegression(max_iter=3000), Zr, yr, cv=skf).mean()
    hn = cross_val_score(make_pipeline(StandardScaler(),
                                       PCA(n_components=10, random_state=RANDOM_STATE),
                                       LogisticRegression(max_iter=3000)),
                         Xr, yr, cv=skf).mean()
    print(f"  {n_r:>6} {lk:>19.4f} {hn:>17.4f} {lk - hn:>+12.4f}")

print()
print("Not even consistently positive. So put PCA in the Pipeline for the reasons that")
print("actually hold:")
print()
print("  1. CORRECTNESS AT SERVE TIME. The fitted rotation (components_ and mean_) is part")
print("     of the model and must ship with it. A pipeline guarantees that; a notebook cell")
print("     that transformed a DataFrame in place does not.")
print("  2. It is required for tuning k inside a grid search at all (2.4).")
print("  3. It costs nothing and removes a whole category of doubt.")
print()
print("And spend your leakage-hunting effort where it pays: any step that touches y.")
print("Target encoding, supervised feature selection, LDA, resampling, and threshold")
print("tuning are where the double-digit inflation lives (NB-00 section 2).")

## 3.5 Whitening

`whiten=True` divides each component by its standard deviation, so all components come out
with unit variance. It discards the relative-importance information that PCA just computed —
which is sometimes exactly right and sometimes exactly wrong.

In [ ]:
Xb_s = StandardScaler().fit_transform(Xb)
for w in [False, True]:
    Z = PCA(n_components=5, whiten=w, random_state=RANDOM_STATE).fit_transform(Xb_s)
    print(f"  whiten={str(w):<6} component variances: {np.round(Z.var(axis=0), 4)}")

print(f"\n  {'model':<44} {'CV accuracy':>12}")
print("  " + "-" * 58)
for w in [False, True]:
    for name, clf in [("KNN", KNeighborsClassifier()),
                      ("logistic regression", LogisticRegression(max_iter=3000))]:
        s = cross_val_score(make_pipeline(StandardScaler(),
                                          PCA(n_components=5, whiten=w,
                                              random_state=RANDOM_STATE), clf),
                            Xb, yb, cv=skf).mean()
        print(f"  {f'PCA(5, whiten={w}) -> {name}':<44} {s:>12.4f}")

print()
print("Note the two rows disagree, which is the point of showing both.")
print()
print("Whitening HURTS KNN and HELPS logistic regression here - and that is the opposite of")
print("the advice usually given, which is 'whiten for distance-based methods like KNN'.")
print()
print("The mechanism explains both directions. Whitening rescales every component to unit")
print("variance, which promotes the low-variance components to equal footing with PC1:")
print("  - For KNN, those trailing components are mostly NOISE on this data, and giving")
print("    noise an equal vote in the distance makes the neighbourhoods worse.")
print("  - For logistic regression, each component gets its own coefficient anyway, so the")
print("    scaling does not change what the model CAN represent - it changes the geometry")
print("    the optimiser and the L2 penalty see, and here that helps slightly.")
print()
print("So the real rule is not 'whiten for KNN'. It is: whitening is worth it when the")
print("low-variance components carry signal, and harmful when they carry noise - which is a")
print("property of your DATA, not of your model. You cannot know it in advance.")
print()
print("Treat it as a hyperparameter and tune it. 2.4 puts it in the grid for exactly this")
print("reason, and the grid there chose whiten=False.")

***
# Part 4 - Tough questions

***

### Q1. PCA is described both as "maximise variance" and as "minimise reconstruction error". Which is it?

<details><summary>Answer</summary>

**Both. They are the same objective, exactly — not merely similar.**

For centred data, Pythagoras splits each point's squared length into the part inside your
subspace and the part perpendicular to it:

$$ \lVert x \rVert^2 = \lVert \hat{x} \rVert^2 + \lVert x - \hat{x} \rVert^2 $$

Summing over all points: **total variance = variance kept + reconstruction error.** The total
is a property of the data and does not depend on your choice of subspace, so maximising the
first term and minimising the second are one operation.

§1.2 verifies it numerically on digits: the sum is constant to **4.6e-13** across every $k$
from 1 to 64, and equals the total variance.

**Why it matters that you know this:**

- It tells you PCA optimises **squared** error, which is why it is sensitive to outliers (a
  point far away dominates the sum) and to feature scale (§1.5).
- It explains why PCA is the *optimal* linear compression under squared loss — no other
  $k$-dimensional linear subspace reconstructs better. That is the Eckart–Young theorem.
- It is the bridge to the SVD formulation (§1.4), and to autoencoders: a linear autoencoder
  trained with MSE recovers the PCA subspace.

</details>

***

### Q2. Why does sklearn use SVD instead of eigendecomposing the covariance matrix?

<details><summary>Answer</summary>

**Numerical conditioning.** Forming $X^\top X$ squares the condition number.

The condition number measures how much a matrix amplifies relative error. float64 gives you
about 16 significant digits. §1.4 measures the squaring directly:

| cond(X) | cond(XᵀX) |
|---|---|
| 1e3 | 1e6 |
| 1e6 | 1e12 |
| 1e8 | ~1e16 — **no precision left** |

A design matrix with condition number $10^8$ is unremarkable in real data (two nearly
collinear features will do it). Eigendecomposing its covariance matrix means working at the
edge of float64.

The SVD $X_c = U\Sigma V^\top$ gives everything you need without ever forming $X^\top X$: the
rows of $V^\top$ are the components, and $\sigma_i^2/(n-1)$ are the variances. §1.4 confirms
both match sklearn to ~1e-13.

**Two related practical points:**

- **`TruncatedSVD` is not PCA.** It skips centring, which is what lets it work on sparse
  matrices without densifying them. On text (TF-IDF), that is what you want — this is LSA.
  On dense data, use `PCA`.
- **`svd_solver="randomized"`** approximates the top $k$ components far faster than the full
  decomposition, and is sklearn's default when the matrix is large and $k$ is small.

</details>

***

### Q3. Do you have to scale before PCA?

<details><summary>Answer</summary>

**Centring: always, and sklearn does it for you. Scaling: it depends, and sklearn does not.**

**Centring is not optional.** PCA is defined on the covariance of centred data. §1.5 shows what
happens without it: the first "component" points at the **mean** rather than the direction of
spread — measured at **0.05°** from the mean direction. sklearn's `PCA` centres internally, so
this only bites if you write the SVD yourself.

**Scaling is a modelling decision**, because variance has units. A feature in grams has
$10^6$ times the variance of the same feature in kilograms, and PCA will hand it PC1 for that
reason alone.

§1.5 measures it on wine: unscaled, PC1 reports **99.8%** of the variance and one feature
(`proline`, whose raw variance dwarfs everything) holds **96.9%** of PC1's loading. Downstream
accuracy drops from **0.9552 to 0.6965**.

**Scale when** features have different units — almost always for tabular data. Use
`StandardScaler` inside the pipeline.

**Do not scale when** features are already commensurable: pixels on a shared 0–255 scale, or
one spectroscopic assay. Scaling there amplifies the quietest, noisiest channels. Digits (§2)
is exactly this case.

A useful way to remember it: PCA on the **correlation** matrix is PCA on scaled data; PCA on
the **covariance** matrix is PCA on raw data. You are choosing between them.

</details>

***

### Q4. How do you choose k?

<details><summary>Answer</summary>

Three methods, in increasing order of trustworthiness.

1. **Scree plot / elbow.** Plot eigenvalues, look for where the curve flattens. Fast,
   subjective, and often there is no clear elbow — §1.6's digits plot does not have one.
2. **Cumulative variance threshold.** `PCA(n_components=0.95)`. §1.6: digits needs 29 of 64
   components for 95%, 41 for 99%. Convenient, and it is a statement about **the data**, not
   about your task.
3. **Cross-validate it.** If you have labels and a downstream model, $k$ is a hyperparameter.
   §2.4 tunes it alongside `whiten` and `n_neighbors` in one grid.

**Use method 3 when you can.** Methods 1 and 2 cannot know what your model needs — §1.8 builds
a case where the 95% rule discards the only informative direction.

**Two observations from the notebook worth carrying:**

- On digits, the 95% rule landed within a rounding error of the cross-validated optimum. The
  heuristic is often fine; the point is that you can *check* rather than assume.
- §1.6's KNN accuracy is **flat** from about k=20 to k=64. Dropping half the components costs
  nothing measurable — which is the realistic version of the "PCA denoises" claim. You get the
  dimensionality reduction for free; you should not expect it to make the model better.

**If there are no labels** (compression, visualisation, clustering input), you have no
downstream criterion, so a variance threshold is a reasonable default. For visualisation the
answer is 2 or 3, and the question does not arise.

</details>

***

### Q5. Your top component explains 95% of the variance and your classifier still fails. What happened?

<details><summary>Answer</summary>

**PCA is unsupervised. Variance is not relevance.**

PCA finds directions that describe $X$. Nothing forces the biggest of those to relate to $y$.
§1.8 constructs the failure explicitly:

| kept | share of variance | CV accuracy |
|---|---|---|
| PC1 | **98.9%** | **0.5467** — barely above a coin flip |
| PC2 | 1.1% | **1.0000** |

The high-variance direction was a noise feature with a large spread; the signal lived in a
low-variance direction. "Keep 95% of the variance" would have kept only PC1 and deleted the
entire signal — and the explained-variance report would have looked like a textbook success,
with 99% of the variance in a single component.

**How to diagnose:** correlate each component with $y$, or score each one alone, as §1.8 does
for breast cancer. If low-ranked components are the predictive ones, PCA is ranking against you.

**How to fix:**

- **Cross-validate $k$** rather than using a variance threshold (Q4).
- **Use LDA** if you have labels and want a supervised reduction (§3.3): on breast cancer,
  LDA(1) scores **0.9684** against PCA(1)'s **0.9104**.
- **Use supervised feature selection**, or a model that does its own selection (a tree
  ensemble ignores irrelevant columns for free — NB-04).
- **Do not reduce at all.** Many models handle moderate dimensionality fine.

**The honest caveat**, which §1.8 also measures: on real data, variance and predictiveness
usually *do* line up — PC1 is the most predictive component on breast cancer. This is a failure
mode to check for, not a reason to distrust PCA generally.

</details>

***

### Q6. Are principal components interpretable? Can you name them?

<details><summary>Answer</summary>

**Sometimes, carefully, and much less often than people claim.**

A component is a **unit vector in the original feature space** with one loading per feature.
It is a direction, not a new measurement. You interpret it by reading which features have
large-magnitude loadings and what signs they carry.

§1.7 makes this concrete: on digits, each component *is* an 8×8 image — a pattern of "more
intensity here, less there".

**Four reasons to be cautious:**

1. **Sign is arbitrary.** $v$ and $-v$ describe the same axis. §1.3 finds sklearn's convention
   differing from a raw eigendecomposition on **30 of the 61** well-determined components — so
   roughly half, as you would expect from an arbitrary choice. A component flipping between
   runs or library versions means nothing, so never build a story on the direction of a sign.
2. **Components are only individually determined when eigenvalues are separated.** Where two
   eigenvalues are nearly equal, only the *plane* they span is determined — the individual
   axes within it are arbitrary. §1.3 flags digits' three zero eigenvalues, whose eigenvectors
   are an arbitrary basis of a degenerate subspace.
3. **Orthogonality is a mathematical constraint, not a claim about the world.** PC2 is
   perpendicular to PC1 because that is how PCA is defined, not because the underlying factors
   are independent. Real factors are usually correlated — which is precisely why **factor
   analysis** with oblique rotation exists.
4. **Every feature contributes**, so "PC1 = size" is a shorthand for a weighted sum of all of
   them.

**When interpretation is genuinely safe:** the eigenvalue is well separated from its
neighbours, a few loadings clearly dominate, and the pattern is stable when you refit on a
bootstrap sample. Check that last one — it takes five lines and settles the question.

</details>

***

### Q7. Does PCA reduce the number of features you need to collect?

<details><summary>Answer</summary>

**No — and this is the misconception that costs money.**

PCA is a **rotation**. Computing PC1 for a new sample requires **every original feature**, because
PC1 is a weighted sum of all of them:

$$ z_1 = w_{11}x_1 + w_{12}x_2 + \dots + w_{1p}x_p $$

§1.7 shows it: digits' PCA needs all 64 pixels to produce even a single score. You have reduced
the **dimensionality of the representation**, not the number of measurements.

So if your goal is any of:

- stop paying for an expensive sensor or lab assay,
- shorten a survey,
- reduce the data your API must accept,

…then PCA does not help at all. You want **feature selection**: `SelectKBest`, recursive
feature elimination, L1 regularisation (NB-01), or a tree ensemble's importances (NB-04 — with
the caveats there).

**What PCA does buy you:**

- Smaller downstream models, and faster ones **when the dimensionality is large enough to
  matter**. Note §2.4 measures PCA making the digits pipeline slightly *slower* — fitting the
  rotation costs more than it saves at 64 features. The speed argument needs scale.
- Denoising — §1.6's KNN accuracy *improves* when trailing components are dropped.
- Decorrelated inputs, which some methods want.
- Compression of stored data (§2.2), noting that you must also store the basis: at $k=64$ the
  "compression" is **0.97×**, i.e. larger than the original.
- Visualisation.

</details>

***

### Q8. When would you use t-SNE or UMAP instead of PCA?

<details><summary>Answer</summary>

**For looking at data. Essentially never as a step in a model.**

They optimise a fundamentally different objective — preserve *local neighbourhoods*, let global
structure go — and §3.2 measures the trade on digits:

| | 10-NN neighbours kept (local) | distance correlation (global) |
|---|---|---|
| PCA(2) | 25.3% | **0.60** |
| t-SNE(2) | **73.2%** | 0.49 |

t-SNE keeps nearly three times as many true neighbours. Its clusters are real. But it is
*worse* at global distances, which is why **the gaps between clusters and their relative sizes
carry no information**.

**Why they cannot be preprocessing steps:**

1. **`TSNE` has no `transform` method** (§3.2 checks: `False`). You cannot project new data
   into an existing embedding, so there is no train/serve boundary. This alone is decisive.
   UMAP *does* offer `transform`, which makes it less obviously disqualified — but the rest
   still applies.
2. **The output depends on hyperparameters** in ways that change the picture — §3.2 varies
   perplexity across 5/30/50 and the embedding quality and scale both move.
3. **It is slow and scales badly** — §3.2 times it at hundreds of times slower than PCA, growing
   with $n$.
4. **The axes are meaningless.** There is no "t-SNE component 1" to interpret or reuse.

**Use them to:** find clusters worth investigating, spot mislabelled data, present a result,
sanity-check that classes are distinguishable at all. Then model on PCA components or the raw
features.

**And when reading someone else's t-SNE plot:** clusters mean something, distances between
them do not, and shapes are an artifact of the optimiser.

</details>

***

### Q9. PCA gives poor results on data you know has structure. What might be wrong?

<details><summary>Answer</summary>

Work through these in order:

1. **You did not scale, and your features have different units** (§1.5, Q3). The single most
   common cause. Symptom: PC1 explains a suspiciously large share of the variance and its
   loading is dominated by one high-variance feature.
2. **The structure is non-linear.** PCA is a rotation and cannot unfold curved manifolds. §3.1:
   on concentric circles, PCA(2) keeps **100%** of the variance and improves a linear
   classifier by **nothing**, while KernelPCA makes the problem trivially separable. Try
   `KernelPCA`, or t-SNE/UMAP if you only need to look.
3. **The signal is in a low-variance direction** (§1.8, Q5). Diagnose by scoring components
   individually.
4. **Outliers are steering it.** PCA minimises *squared* error (Q1), so a few extreme points
   can capture PC1 entirely. Check for them; consider `RobustScaler`, winsorising, or a robust
   PCA variant.
5. **Your features are not continuous.** PCA assumes meaningful covariance. One-hot columns,
   counts, and ordinal codes violate that. Consider MCA for categorical data, or
   `TruncatedSVD` for sparse counts.
6. **There genuinely is no low-dimensional linear structure.** If the scree plot is flat, the
   data really is high-dimensional — that is an answer, not a failure.

</details>

***

### Q10. What is the relationship between PCA and an autoencoder?

<details><summary>Answer</summary>

**A linear autoencoder trained with squared error learns the PCA subspace.**

An autoencoder compresses input to a bottleneck and reconstructs it, minimising reconstruction
error. If the encoder and decoder are linear and the loss is MSE, that is *exactly* the
objective of Q1 — so the optimum spans the same subspace as the top $k$ principal components.

**Two differences worth knowing:**

- The autoencoder does not recover the components in order, or orthogonally. It finds *a* basis
  for the same subspace, not *the* PCA basis. (You can recover ordering by adding constraints.)
- PCA has a closed-form solution via SVD; the autoencoder needs gradient descent to reach the
  same place, more slowly and with no guarantee of the global optimum.

**Where autoencoders earn their keep** is the non-linear case: add non-linear activations and
they learn curved manifolds that PCA cannot (§3.1). That is the same gap KernelPCA addresses,
by a different route — and unlike KernelPCA, autoencoders scale to large $n$ because they never
form an $n \times n$ matrix.

**The practical advice:** if a linear method suffices, use PCA — it is deterministic, fast, has
no hyperparameters beyond $k$, and cannot fail to converge. Reach for an autoencoder when you
have established that you need non-linearity and you have the data to fit one.

</details>

***

### Q11. How does PCA behave when p > n — more features than samples?

<details><summary>Answer</summary>

It still works, with a hard limit and a real statistical caveat.

**The limit:** the covariance matrix of $n$ centred points has rank at most $n-1$, so there are
at most $n-1$ non-zero eigenvalues. With 50 samples and 10,000 genes you can extract at most
**49** components, regardless of $p$. sklearn caps `n_components` at
`min(n_samples, n_features)` accordingly.

**The efficiency:** never form the $p \times p$ covariance matrix — that is $10^8$ entries for
10,000 genes. SVD on the $n \times p$ data costs $O(n^2 p)$ instead, and
`svd_solver="randomized"` is faster still. sklearn picks sensibly by default.

**The caveat that actually matters:** with $p \gg n$, the estimated components are **noisy**.
Sample eigenvalues are biased upward and eigenvectors are unstable — a phenomenon studied under
random matrix theory (the Marchenko–Pastur distribution describes what pure noise looks like,
so you can test whether an eigenvalue exceeds what noise alone would produce).

**So:** check stability before interpreting anything. Refit on bootstrap samples and see whether
the loadings persist. If they do not, you are reading noise — and in $p \gg n$ regimes,
that is the default state rather than the exception.

This is the standard situation in genomics, spectroscopy and neuroimaging, which is why those
fields are careful about it and general tutorials are not.

</details>

***

### Q12. Your CV score with PCA is 0.95 and production is 0.70. Is the PCA to blame?

<details><summary>Answer</summary>

**Probably not directly** — but there are three PCA-specific things to rule out, and one common
misdiagnosis.

**The misdiagnosis first.** Many people reach for "I fitted PCA outside the CV loop, so it
leaked". §3.4 measures that on pure noise: fitting PCA on all the data inflates accuracy by
**−0.015** — nothing at all. PCA never sees $y$, so it has no label information to smuggle. If
you *did* fit something outside the loop and see a 25-point gap, look for a **supervised** step:
`SelectKBest` on the same noise fabricates **78.5%** accuracy. Target encoding, LDA, resampling
and threshold tuning are the real culprits (NB-00 §2).

**The genuinely PCA-specific causes:**

1. **The fitted rotation did not ship.** `components_` and `mean_` are part of the model. If
   production re-fits PCA on incoming data, it computes a *different* rotation and the
   downstream model receives coordinates in a coordinate system it never saw. This produces
   exactly this symptom and is the most likely real cause.
2. **Distribution shift moved the components.** PCA describes the training data's covariance.
   If production data has a different covariance structure, the fixed rotation is projecting
   onto directions that no longer describe it well. Monitor the **reconstruction error** of
   incoming data — it is one line, it needs no labels, and a rising value is a strong
   distribution-shift alarm. This is a genuinely useful diagnostic that PCA gives you free.
3. **Feature order or units changed.** PCA is a dot product against `components_`; permuting or
   rescaling the input columns silently produces garbage rather than an error.

**The general causes** — temporal drift, a train/test split that was not representative,
overfitting the hyperparameter search — are more likely than any of these and should be checked
too (NB-00 Part 9).

</details>

***

## Coding challenges

### Challenge 1 — eigenfaces, and the ethics of a 20-dimensional face

`fetch_olivetti_faces()` gives 400 face images at 64×64 = 4,096 pixels.

1. Run PCA and display the mean face and the first 16 components as images. These are the
   classic "eigenfaces" of Turk & Pentland (1991).
2. Reconstruct several faces at k = 5, 20, 50, 150. At what k does a face become *recognisable
   as a specific person* rather than merely face-like? Note that is a different question from
   "when does reconstruction error get small".
3. Build a face classifier: `PCA(k) → LogisticRegression`, sweeping k. Plot accuracy against k
   and against fit time.
4. This dataset is $p \gg n$ — 4,096 features, 400 samples. Confirm Q11's rank limit
   empirically: how many non-zero eigenvalues are there?
5. Project a face that is **not** in the dataset (or a non-face image) into the space and
   reconstruct it. The reconstruction error is much higher — this is Q12's monitoring
   diagnostic, and it is also how eigenface-based detection worked.

***

### Challenge 2 — find the crossover where PCA starts helping

PCA denoises (§1.6) but discards information. Map out when the trade pays.

1. Generate `make_classification(n_samples=500, n_features=p, n_informative=10)` for
   p ∈ {20, 50, 100, 500, 1000}, holding the informative count fixed at 10.
2. For each, compare: raw features, PCA(10), PCA(0.95), and `SelectKBest(k=10)`, each feeding
   logistic regression and KNN.
3. Plot accuracy against $p$ for each method. At what $p$ does PCA start beating raw features,
   and is the crossover the same for both models? Explain using NB-07 §1.5.
4. Now repeat with `n_informative=10, n_redundant=50`. Redundant features are linear
   combinations of informative ones — exactly what PCA should absorb. Does it?
5. Which method wins overall, and what does that tell you about reaching for PCA by reflex?

***

### Challenge 3 — how stable are your components?

Q6 and Q11 both say to check stability before interpreting. Build the check.

1. On wine (scaled), fit PCA. Record the loadings of PC1 and PC2.
2. Bootstrap: resample rows with replacement 200 times, refit PCA each time, and collect the
   loadings — **aligning signs each time** against the reference fit, or your averages will
   cancel to zero.
3. Plot each feature's loading on PC1 with a bootstrap confidence interval. Which loadings are
   distinguishable from zero?
4. Repeat on a subsample of 30 rows. The intervals should widen dramatically. This is Q11's
   instability, produced on demand.
5. Now measure how often the *order* of PC2 and PC3 swaps between bootstrap fits, and compare
   that to the gap between their eigenvalues. This is the concrete version of "only interpret
   components whose eigenvalues are well separated".

***
# Part 5 - Five datasets to practise on

| # | Dataset | Shape | The skill it forces | Difficulty |
|---|---|---|---|---|
| 1 | **Iris** | 150 × 4 | The whole method small enough to check by hand | ★☆☆☆☆ |
| 2 | **Wine** | 178 × 13 | **Scaling changes everything**; loading interpretation | ★★☆☆☆ |
| 3 | **Olivetti faces** | 400 × 4,096 | Eigenfaces; **p ≫ n**; visual reconstruction | ★★★☆☆ |
| 4 | **20 Newsgroups TF-IDF** | 11k × 100k+ | `TruncatedSVD` / LSA on **sparse** data | ★★★★☆ |
| 5 | **Single-cell RNA-seq** | ~3k × ~20k | PCA as a mandatory pipeline step in real science | ★★★★★ |

In [ ]:
print("bundled with sklearn:\n")
print(f"  {'dataset':<18} {'rows':>7} {'cols':>7} {'max components':>16}")
print("  " + "-" * 52)
for label, loader in [("iris", load_iris), ("wine", load_wine),
                      ("breast cancer", load_breast_cancer)]:
    b = loader()
    n_r, n_c = b.data.shape
    print(f"  {label:<18} {n_r:>7} {n_c:>7} {min(n_r, n_c):>16}")
print(f"  {'digits':<18} {X_dig.shape[0]:>7} {X_dig.shape[1]:>7} "
      f"{min(X_dig.shape):>16}")

print("\nHow many components does 95% of the variance need, on scaled data?")
for label, loader in [("iris", load_iris), ("wine", load_wine),
                      ("breast cancer", load_breast_cancer)]:
    X_, _ = loader(return_X_y=True)
    p = PCA(n_components=0.95, random_state=RANDOM_STATE).fit(StandardScaler().fit_transform(X_))
    print(f"  {label:<18} {p.n_components_:>2} of {X_.shape[1]:<3} "
          f"({p.n_components_ / X_.shape[1]:.0%} of the original dimensions)")

print("\nOlivetti faces (~4 MB) and 20 Newsgroups download on first use; loaders are in the")
print("briefs below. Single-cell data needs scanpy and is a genuinely large download.")

### 1. Iris — small enough to verify by hand

```python
from sklearn.datasets import load_iris
X, y = load_iris(return_X_y=True)
```

Four features, 150 samples. The point is that you can check every step.

1. Compute the covariance matrix by hand, eigendecompose it, and confirm against
   `PCA().fit(X)` — including the sign-alignment step from §1.3.
2. Fit PCA with and without scaling. Iris features are all in centimetres, so this is the case
   where *not* scaling is defensible. Does it change the answer? Does it change it usefully?
3. Plot PC1 vs PC2 coloured by species. Two species overlap. Now fit LDA(2) and plot that —
   §3.3's argument, on a picture.
4. PC1 will explain ~92% of the variance. Read its loadings: what does it correspond to, in
   plain language? Then check stability by bootstrapping (Challenge 3).

***

### 2. Wine — scaling changes the answer

```python
from sklearn.datasets import load_wine
X, y = load_wine(return_X_y=True)
```

13 chemical measurements on wildly different scales. §1.5 uses it for exactly this.

1. Reproduce §1.5's comparison, then look at the full loading vector of PC1 in both cases.
   Unscaled, one feature holds 96.9% of it — confirm which and why.
2. How many components for 95% of the variance, scaled versus unscaled? The unscaled answer
   will be absurd. Explain it.
3. Build `scale → PCA(k) → classifier` and tune k by CV. Compare against the classifier on all
   13 features. Does PCA help here at all?
4. Interpret PC1 and PC2's loadings on the scaled data. Group the features by what they
   measure — phenolic compounds, colour, alcohol. Do the components align with those groups?
5. Compare against NB-07 §1.2, which used the same dataset to show KNN's scaling trap. Both
   methods break on unscaled wine, for the same underlying reason. State that reason precisely.

***

### 3. Olivetti faces — eigenfaces

```python
from sklearn.datasets import fetch_olivetti_faces
faces = fetch_olivetti_faces()          # 400 images, 64x64, 40 people
```

The dataset PCA became famous on. See Challenge 1 for the full brief.

1. Mean face plus the first 16 eigenfaces, displayed as images.
2. Reconstruction quality against k, both as RMSE and visually.
3. A face classifier over PCA components; sweep k.
4. Confirm the $p \gg n$ rank limit from Q11.
5. Reconstruction error as an out-of-distribution detector (Q12).

***

### 4. 20 Newsgroups — sparse data, and why it needs a different tool

```python
from sklearn.datasets import fetch_20newsgroups_vectorized
data = fetch_20newsgroups_vectorized(subset="train", remove=("headers","footers","quotes"))
```

100,000+ sparse TF-IDF features. **`PCA` will not work here** — find out why before reaching
for `TruncatedSVD`.

1. Try `PCA()` on the sparse matrix. Read the error. Why does centring destroy sparsity, and
   what would densifying cost in memory? Compute that number.
2. Use `TruncatedSVD(n_components=100)` instead. This is **Latent Semantic Analysis**.
3. Inspect the top terms loading on the first few components — do they look like topics?
4. Compare `TruncatedSVD(100) → LogisticRegression` against the raw TF-IDF baseline from
   NB-08 §2.5, on accuracy *and* fit time. Does reduction help on text?
5. Compare against `NMF`, which forces non-negative loadings and therefore additive,
   parts-based components. Which produces more interpretable topics, and why would
   non-negativity matter for that?

***

### 5. Single-cell RNA-seq — PCA as a mandatory step

```python
# pip install scanpy
import scanpy as sc
adata = sc.datasets.pbmc3k()      # ~2,700 cells x ~32,000 genes
```

Real science where PCA is not optional — it is step three of every standard pipeline.

1. Follow the standard preprocessing: filter, normalise, log-transform, select highly variable
   genes. Note that the log transform happens **before** PCA, and think about why.
2. Run PCA to 50 components. Plot the scree — how many components carry real signal?
3. Build a neighbour graph **on the PCA components**, not the raw genes, then cluster
   (Leiden) and visualise with UMAP. Notice that PCA is the denoising step that makes the
   rest work — NB-10 covers the clustering.
4. This is $p \gg n$ (Q11). How many components could you extract in principle?
5. Compare clustering on raw genes against clustering on PCA components. The difference is
   the entire justification for the standard pipeline.

***
# Part 6 - Reading the literature

PCA is old — older than computers — and its literature splits neatly into the founding papers,
the numerical-linear-algebra results that made it practical, and the modern non-linear
successors.

## Start here

**1. [On Lines and Planes of Closest Fit to Systems of Points in Space](https://www.tandfonline.com/doi/abs/10.1080/14786440109462720)** —
Karl Pearson, *Philosophical Magazine* 2(11):559–572, **1901**.
> **The original, and it is the reconstruction-error formulation.** Pearson posed PCA as
> finding the line or plane of *closest fit* — minimising perpendicular distance — a full three
> decades before the variance-maximisation framing. Reading it makes Q1's equivalence feel
> inevitable rather than clever. Short, and written before the notation hardened, so it is more
> readable than you expect.

**2. [Analysis of a Complex of Statistical Variables into Principal Components](https://psycnet.apa.org/record/1934-00645-001)** —
Harold Hotelling, *Journal of Educational Psychology* 24:417–441, **1933**.
> The paper that gave the method its name and the variance-maximisation derivation, and
> connected it to the eigenstructure of the covariance matrix. This is the version taught today.

**3. [A Tutorial on Principal Component Analysis](https://arxiv.org/abs/1404.1100)** —
Jonathon Shlens, **2014**. **Free.**
> Not a research paper — the best pedagogical treatment available, and the one to read if any
> of Part 1 felt rushed. Builds from change of basis to the SVD carefully, with the algebra
> written out. Twelve pages.

## The paper behind each section

| Section | Source | Free? |
|---|---|---|
| 1.2 — variance ⇔ reconstruction | **Pearson**, **1901** (reconstruction) and **Hotelling**, **1933** (variance) | 🔍 |
| 1.2 — that PCA is the *optimal* linear compression | **Eckart & Young**, *The approximation of one matrix by another of lower rank*, Psychometrika 1, **1936** | 🔍 |
| 1.3–1.4 — the SVD, and why it beats eig(cov) | **Golub & Van Loan**, *Matrix Computations*, 4th ed., ch. 8 — the standard reference | 🔍 |
| 1.4 — randomized SVD, sklearn's default for large data | **Halko, Martinsson & Tropp**, *Finding Structure with Randomness*, SIAM Review 53(2), **2011** — [arXiv](https://arxiv.org/abs/0909.4061) | ✅ |
| 1.6 — choosing k properly | **Minka**, *Automatic Choice of Dimensionality for PCA*, NeurIPS **2000** — [pdf](https://proceedings.neurips.cc/paper/2000/hash/7503cfacd12053d309b6bed5c89de212-Abstract.html) (this is sklearn's `n_components="mle"`) | ✅ |
| 1.8, Q5 — a probabilistic account of what PCA assumes | **Tipping & Bishop**, *Probabilistic Principal Component Analysis*, JRSS-B 61(3), **1999** — [pdf](https://www.microsoft.com/en-us/research/publication/probabilistic-principal-component-analysis/) | ✅ |
| 3.1 — KernelPCA | **Schölkopf, Smola & Müller**, *Nonlinear Component Analysis as a Kernel Eigenvalue Problem*, Neural Computation 10(5), **1998** | 🔍 |
| 3.2 — **t-SNE** | **van der Maaten & Hinton**, *Visualizing Data using t-SNE*, JMLR 9, **2008** — [pdf](https://www.jmlr.org/papers/volume9/vandermaaten08a/vandermaaten08a.pdf) | ✅ |
| 3.2 — how to read a t-SNE plot without fooling yourself | **Wattenberg, Viégas & Johnson**, *How to Use t-SNE Effectively*, Distill, **2016** — [interactive](https://distill.pub/2016/misread-tsne/) | ✅ |
| 3.2 — UMAP | **McInnes, Healy & Melville**, *UMAP*, **2018** — [arXiv](https://arxiv.org/abs/1802.03426) | ✅ |
| 3.3 — LDA | **Fisher**, *The Use of Multiple Measurements in Taxonomic Problems*, Annals of Eugenics 7, **1936** | 🔍 |
| Q10 — PCA and autoencoders | **Baldi & Hornik**, *Neural Networks and Principal Component Analysis*, Neural Networks 2(1), **1989** | 🔍 |
| Q11 — what noise eigenvalues look like | **Marchenko & Pastur**, **1967**; see also **Johnstone**, *On the distribution of the largest eigenvalue*, Annals of Statistics 29(2), **2001** | 🔍 |
| Challenge 1 — eigenfaces | **Turk & Pentland**, *Eigenfaces for Recognition*, J. Cognitive Neuroscience 3(1), **1991** — [pdf](https://www.face-rec.org/algorithms/PCA/jcn.pdf) | ✅ |

**Legend:** ✅ free at the link · 🔍 search the exact title on
[Google Scholar](https://scholar.google.com)

### If you read only one

**Shlens (2014)**, because it will make everything in Part 1 click, and it is free and short.

Then **Wattenberg, Viégas & Johnson (2016)** — it is interactive, it takes fifteen minutes, and
it will permanently change how you read every t-SNE and UMAP plot you encounter afterwards.
That is an unusually high return for the time.

For the historical pleasure, **Pearson (1901)**: the founding paper of a method you now use in
one line.

***
# Appendix

| Symptom | Cause | Fix |
|---|---|---|
| PC1 explains ~99% of the variance | Unscaled features with different units (§1.5) | `StandardScaler` inside the Pipeline |
| First component points at the data's mean | Forgot to centre — hand-rolled SVD (§1.5) | Centre first, or use sklearn's `PCA` |
| `ValueError` / `TypeError` on a sparse matrix | `PCA` centres, which densifies | `TruncatedSVD` (this is LSA) |
| `MemoryError` on wide data | Formed the $p \times p$ covariance matrix | Use SVD; `svd_solver="randomized"` |
| Components differ in sign between runs | Sign is arbitrary — $v$ and $-v$ are the same axis (§1.3) | Align signs before comparing; never interpret a sign |
| Components change completely on refit | Eigenvalues nearly tied, so only the subspace is determined (§1.3, Q6) | Check eigenvalue gaps; bootstrap for stability |
| Classifier worse after PCA | Signal was in a low-variance direction (§1.8) | Cross-validate k; try LDA; or skip reduction |
| Can only get n−1 components | Rank limit when p > n (Q11) | Expected; not a bug |
| PCA did not help a non-linear problem | PCA is a rotation (§3.1) | `KernelPCA`, autoencoder, or a non-linear model |
| t-SNE embedding has no `transform` | By design — it cannot project new points (§3.2) | Never use t-SNE in a pipeline; use PCA or UMAP |
| Production much worse than CV | The fitted rotation did not ship, or the input columns moved (Q12) | Ship the whole `Pipeline`; monitor reconstruction error |
| "PCA reduced my costs" | It did not — all original features are still required (Q7) | Use feature *selection* if that is the goal |

## Checklist for shipping PCA

Everything in the Foundations checklist, plus:

- [ ] Is PCA **inside the Pipeline**, so the fitted rotation ships with the model?
- [ ] Are features **scaled** first — or have I deliberately decided they are already
      commensurable (§1.5)?
- [ ] Was **k cross-validated** against the actual downstream model, not just chosen by a
      variance threshold (§1.6)?
- [ ] Have I checked that low-variance components are not the predictive ones (§1.8)?
- [ ] If I am interpreting loadings: are the eigenvalues well separated, and are the loadings
      **stable under bootstrap** (Q6)?
- [ ] Am I monitoring **reconstruction error** on incoming data as a distribution-shift alarm
      (Q12)?
- [ ] Have I compared against **no reduction at all**, so I know what PCA bought?
- [ ] If the goal was to collect fewer features: am I aware PCA does not do that (Q7)?

## Where to go next

| Notebook | Why it follows |
|---|---|
| `kmeans_zero_to_hero.ipynb` | The other core unsupervised method, and PCA is its standard preprocessing step. Same distance assumptions, same scaling requirement. |
| [`knn_zero_to_hero.ipynb`](knn_zero_to_hero.ipynb) | §1.5 there is the curse of dimensionality this notebook exists to fight; §2.6 used PCA to rescue a distance and only showed the outcome. |
| [`svm_zero_to_hero.ipynb`](svm_zero_to_hero.ipynb) | The kernel trick §3.1 reuses, derived properly. |
| `explainable_ai_zero_to_hero.ipynb` | Q6 is about whether a component *means* anything. That notebook is the same question asked of whole models. |

See [`README.md`](README.md) for the full roster, the reading order and the suggested paths.